# Data Quality & Integrity Checks

This notebook validates the integrity of the raw datasets and checks whether the relationships between tables behave as expected.

The checks focus on keys, relationships, grain, and potential issues that could affect downstream analysis.

In [45]:
from pathlib import Path

import pandas as pd

In [46]:
DATA_DIR = Path("../data/raw")

datasets = {
    file.stem: pd.read_csv(file)
    for file in sorted(DATA_DIR.glob("*.csv"))
}

In [47]:
customers = datasets["olist_customers_dataset"]
orders = datasets["olist_orders_dataset"]
order_items = datasets["olist_order_items_dataset"]
payments = datasets["olist_order_payments_dataset"]
reviews = datasets["olist_order_reviews_dataset"]
products = datasets["olist_products_dataset"]
sellers = datasets["olist_sellers_dataset"]
geolocation = datasets["olist_geolocation_dataset"]
category_translation = datasets["product_category_name_translation"]

## Primary Key Checks

Each primary key should be non-null and unique within its table.

In [48]:
primary_keys = {
    "customers": (customers, "customer_id"),
    "orders": (orders, "order_id"),
    "products": (products, "product_id"),
    "sellers": (sellers, "seller_id"),
}

And test:

In [49]:
primary_key_results = []

for table_name, (df, column) in primary_keys.items():
    primary_key_results.append(
        {
            "table": table_name,
            "column": column,
            "null_count": df[column].isna().sum(),
            "duplicate_count": df[column].duplicated().sum(),
            "is_valid": (
                df[column].notna().all()
                and df[column].is_unique
            ),
        }
    )

primary_key_results = pd.DataFrame(primary_key_results)

primary_key_results

,table,column,null_count,duplicate_count,is_valid
0,customers,customer_id,0,0,True
1,orders,order_id,0,0,True
2,products,product_id,0,0,True
3,sellers,seller_id,0,0,True


We expect all `is_valid`s to be True.

## Order Items Key

An order can contain multiple items, so `order_id` is not expected to be unique in `order_items`.

The combination of `order_id` and `order_item_id` should uniquely identify each order item.

In [50]:
order_item_key = ["order_id", "order_item_id"]

order_items.duplicated(subset=order_item_key).sum()

np.int64(0)

If the output is `0`, it means, this combination has no duplicates.

To make the test clearer and more usable in reporting:

In [51]:
order_items_key_check = pd.DataFrame(
    {
        "table": ["order_items"],
        "key": ["order_id + order_item_id"],
        "duplicate_count": [
            order_items.duplicated(subset=order_item_key).sum()
        ],
    }
)

order_items_key_check

,table,key,duplicate_count
0,order_items,order_id + order_item_id,0


We want to see if `order_item_id` alone is unique or not.

In [52]:
order_items["order_item_id"].is_unique

False

It's not a problem that it's `False`.
Incidentally, this is what we expect, because `order_item_id` is the item number within each order, not a global ID.

## Foreign Key Checks

Foreign key values should reference existing records in the related parent table.

In [53]:
def check_foreign_key(child_df, child_column, parent_df, parent_column):
    child_values = child_df[child_column].dropna()
    parent_values = set(parent_df[parent_column].dropna())

    orphan_count = (~child_values.isin(parent_values)).sum()

    return orphan_count

Now the main relationships:

In [54]:
foreign_key_checks = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
]

In [55]:
foreign_key_results = []

table_map = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
}

for child_table, child_column, parent_table, parent_column in foreign_key_checks:
    orphan_count = check_foreign_key(
        table_map[child_table],
        child_column,
        table_map[parent_table],
        parent_column,
    )

    foreign_key_results.append(
        {
            "child_table": child_table,
            "child_column": child_column,
            "parent_table": parent_table,
            "parent_column": parent_column,
            "orphan_count": orphan_count,
            "is_valid": orphan_count == 0,
        }
    )

foreign_key_results = pd.DataFrame(foreign_key_results)

foreign_key_results

,child_table,child_column,parent_table,parent_column,orphan_count,is_valid
0,orders,customer_id,customers,customer_id,0,True
1,order_items,order_id,orders,order_id,0,True
2,order_items,product_id,products,product_id,0,True
3,order_items,seller_id,sellers,seller_id,0,True
4,payments,order_id,orders,order_id,0,True
5,reviews,order_id,orders,order_id,0,True


We expect all `is_valid`s to be True.

## Relationship Cardinality & Grain

This section examines the cardinality of key relationships between tables and verifies the grain of the main transactional datasets.

Understanding these relationships helps prevent incorrect joins and double counting in downstream analysis.

### Orders per Customer

First, checking how many orders each `customer_id` has:

In [56]:
orders_per_customer = (
    orders.groupby("customer_id")
    .size()
    .rename("order_count")
)

orders_per_customer.describe()

count    99441.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: order_count, dtype: float64

Next, the distribution of the number of orders:

In [57]:
orders_per_customer.value_counts().sort_index()

order_count
1    99441
Name: count, dtype: int64

And to find out how many customers have more than one order:

In [58]:
{
    "customers_with_one_order": (orders_per_customer == 1).sum(),
    "customers_with_multiple_orders": (orders_per_customer > 1).sum(),
    "max_orders_per_customer": orders_per_customer.max(),
}

{'customers_with_one_order': np.int64(99441),
 'customers_with_multiple_orders': np.int64(0),
 'max_orders_per_customer': np.int64(1)}

This check is based on `customer_id`. We won't go into `customer_unique_id` yet, because we're currently checking the cardinality of `orders.customer_id → customers.customer_id` relationship.

### Items per Order

Now `orders → order_items` relationship:

In [59]:
items_per_order = (
    order_items.groupby("order_id")
    .size()
    .rename("item_count")
)

items_per_order.describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: item_count, dtype: float64

Distribution:

In [60]:
items_per_order.value_counts().sort_index()

item_count
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64

And a few important values:

In [61]:
{
    "orders_with_one_item": (items_per_order == 1).sum(),
    "orders_with_multiple_items": (items_per_order > 1).sum(),
    "max_items_per_order": items_per_order.max(),
}

{'orders_with_one_item': np.int64(88863),
 'orders_with_multiple_items': np.int64(9803),
 'max_items_per_order': np.int64(21)}

This shows us that `order_id` is intentionally repeated in `order_items` and why joining `orders` with `order_items` increases the number of rows.

### Payments per Order

Now the same process for payments:

In [62]:
payments_per_order = (
    payments.groupby("order_id")
    .size()
    .rename("payment_count")
)

payments_per_order.describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
Name: payment_count, dtype: float64

Distribution:

In [63]:
payments_per_order.value_counts().sort_index()

payment_count
1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [64]:
{
    "orders_with_one_payment": (payments_per_order == 1).sum(),
    "orders_with_multiple_payments": (payments_per_order > 1).sum(),
    "max_payments_per_order": payments_per_order.max(),
}

{'orders_with_one_payment': np.int64(96479),
 'orders_with_multiple_payments': np.int64(2961),
 'max_payments_per_order': np.int64(29)}

This section is very important, because we already know that `order_payments` does not necessarily have one row per order.

### Reviews per Order

In [65]:
reviews_per_order = (
    reviews.groupby("order_id")
    .size()
    .rename("review_count")
)

reviews_per_order.describe()

count    98673.000000
mean         1.005584
std          0.075060
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
Name: review_count, dtype: float64

In [66]:
reviews_per_order.value_counts().sort_index()

review_count
1    98126
2      543
3        4
Name: count, dtype: int64

In [67]:
{
    "orders_with_one_review": (reviews_per_order == 1).sum(),
    "orders_with_multiple_reviews": (reviews_per_order > 1).sum(),
    "max_reviews_per_order": reviews_per_order.max(),
}

{'orders_with_one_review': np.int64(98126),
 'orders_with_multiple_reviews': np.int64(547),
 'max_reviews_per_order': np.int64(3)}

This one is very important for the continuation of the project, because if some orders have multiple reviews, we can no longer say without making a decision that each order is equal to one review.

### A Direct Grain Check for `order_items`

Now a very simple test to verify grain:

In [68]:
order_item_counts = (
    order_items.groupby(["order_id", "order_item_id"])
    .size()
)

order_item_counts.value_counts()

1    112650
Name: count, dtype: int64

We expect to have only the value `1`.

For a more explicit check:

In [69]:
order_item_counts.max() == 1

np.True_

`True` means each (order_id, order_item_id) combination has exactly one row.

### Cardinality Summary

Finally, let's create a summary table so that the results of this section can be seen in one place:

In [70]:
cardinality_summary = pd.DataFrame(
    [
        {
            "relationship": "Customer → Orders",
            "child_table": "orders",
            "group_key": "customer_id",
            "max_child_rows": orders_per_customer.max(),
            "child_rows_with_multiple": (orders_per_customer > 1).sum(),
        },
        {
            "relationship": "Order → Order Items",
            "child_table": "order_items",
            "group_key": "order_id",
            "max_child_rows": items_per_order.max(),
            "child_rows_with_multiple": (items_per_order > 1).sum(),
        },
        {
            "relationship": "Order → Payments",
            "child_table": "payments",
            "group_key": "order_id",
            "max_child_rows": payments_per_order.max(),
            "child_rows_with_multiple": (payments_per_order > 1).sum(),
        },
        {
            "relationship": "Order → Reviews",
            "child_table": "reviews",
            "group_key": "order_id",
            "max_child_rows": reviews_per_order.max(),
            "child_rows_with_multiple": (reviews_per_order > 1).sum(),
        },
    ]
)

cardinality_summary

,relationship,child_table,group_key,max_child_rows,child_rows_with_multiple
0,Customer → Orders,orders,customer_id,1,0
1,Order → Order Items,order_items,order_id,21,9803
2,Order → Payments,payments,order_id,29,2961
3,Order → Reviews,reviews,order_id,3,547


### Cardinality Findings

- `customer_id` has a one-to-one relationship with `orders` in this dataset. Customer-level analysis should use `customer_unique_id` instead.
- `orders` and `order_items` have a one-to-many relationship.
- `orders` and `order_payments` have a one-to-many relationship.
- `orders` and `order_reviews` can have multiple review records per order.
- Directly joining multiple one-to-many tables on `order_id` can multiply rows and lead to incorrect aggregations.

## Timestamp Consistency

This section checks whether order timestamps follow a logically consistent sequence.

Converting `orders` time data to datetime:

In [71]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

Examining the logical order of the steps:

__Approval should not come before purchase.__

In [72]:
approval_before_purchase = (
    orders["order_approved_at"].notna()
    & (
        orders["order_approved_at"]
        < orders["order_purchase_timestamp"]
    )
)

approval_before_purchase.sum()

np.int64(0)

__Carrier delivery should not come before purchase.__

In [73]:
carrier_before_purchase = (
    orders["order_delivered_carrier_date"].notna()
    & (
        orders["order_delivered_carrier_date"]
        < orders["order_purchase_timestamp"]
    )
)

carrier_before_purchase.sum()

np.int64(166)

__Customer delivery should not come before carrier delivery.__

Here, we only check orders that have both timestamp.

In [74]:
customer_before_carrier = (
    orders["order_delivered_customer_date"].notna()
    & orders["order_delivered_carrier_date"].notna()
    & (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    )
)

customer_before_carrier.sum()

np.int64(23)

__Customer delivery should not come before purchase.__


In [75]:
customer_before_purchase = (
    orders["order_delivered_customer_date"].notna()
    & (
        orders["order_delivered_customer_date"]
        < orders["order_purchase_timestamp"]
    )
)

customer_before_purchase.sum()

np.int64(0)

### Checking the delivered status

If the order reaches the `delivered` status, does it have a customer delivery timestamp?

In [76]:
delivered_without_customer_date = (
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].isna()
)

delivered_without_customer_date.sum()

np.int64(8)

Do we have an order whose status is not `delivered` but has a customer delivery date?

In [77]:
not_delivered_with_customer_date = (
    (orders["order_status"] != "delivered")
    & orders["order_delivered_customer_date"].notna()
)

not_delivered_with_customer_date.sum()

np.int64(6)

These two tests are important because we will likely use `order_status` and timestamps together later in our delivery Performance analysis.

### Numeric Sanity Checks

Now let's move on to numerical values.

In [78]:
negative_price = (order_items["price"] < 0).sum()

negative_price

np.int64(0)

In [79]:
negative_freight = (order_items["freight_value"] < 0).sum()

negative_freight

np.int64(0)

In [80]:
negative_payment = (payments["payment_value"] < 0).sum()

negative_payment

np.int64(0)

### Order Item Sequence

`order_item_id` must start at 1.

Let's see if we have a value of zero:

(order_items["order_item_id"] == 0).sum()

In [81]:
order_items["order_item_id"].min()

np.int64(1)

In [82]:
order_items["order_item_id"].max()

np.int64(21)

### Estimated Delivery Date

Is the estimated delivery date before the purchase date?

In [83]:
estimated_before_purchase = (
    orders["order_estimated_delivery_date"]
    < orders["order_purchase_timestamp"]
)

estimated_before_purchase.sum()

np.int64(0)

### Integrity Summary

In [84]:
integrity_results = pd.DataFrame(
    [
        {
            "check": "Approval before purchase",
            "invalid_count": approval_before_purchase.sum(),
        },
        {
            "check": "Carrier delivery before purchase",
            "invalid_count": carrier_before_purchase.sum(),
        },
        {
            "check": "Customer delivery before carrier delivery",
            "invalid_count": customer_before_carrier.sum(),
        },
        {
            "check": "Customer delivery before purchase",
            "invalid_count": customer_before_purchase.sum(),
        },
        {
            "check": "Delivered orders without customer delivery date",
            "invalid_count": delivered_without_customer_date.sum(),
        },
        {
            "check": "Non-delivered orders with customer delivery date",
            "invalid_count": not_delivered_with_customer_date.sum(),
        },
        {
            "check": "Negative item price",
            "invalid_count": negative_price,
        },
        {
            "check": "Negative freight value",
            "invalid_count": negative_freight,
        },
        {
            "check": "Negative payment value",
            "invalid_count": negative_payment,
        },
        {
            "check": "Estimated delivery before purchase",
            "invalid_count": estimated_before_purchase.sum(),
        },
    ]
)

integrity_results

,check,invalid_count
0,Approval before purchase,0
1,Carrier delivery before purchase,166
2,Customer delivery before carrier delivery,23
3,Customer delivery before purchase,0
4,Delivered orders without customer delivery date,8
5,Non-delivered orders with customer delivery date,6
6,Negative item price,0
7,Negative freight value,0
8,Negative payment value,0
9,Estimated delivery before purchase,0


### Findings

Several timestamp inconsistencies were identified in the order data:

- 166 orders have a carrier delivery timestamp earlier than the purchase timestamp.
- 23 orders have a customer delivery timestamp earlier than the carrier delivery timestamp.
- 8 delivered orders do not have a customer delivery timestamp.
- 6 non-delivered orders have a customer delivery timestamp.

These records require further investigation before deciding how they should be handled in downstream analysis.

### Investigating

In [85]:
orders.loc[
    carrier_before_purchase,
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ],
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37
1111,ad133696906f6a78826daa0911b7daec,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37
1329,74e033208dc13a7b8127eb8e73d09b76,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36
1372,a6b58794fd2ba533359a76c08df576e3,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32
1864,5792e0b1c8c8a2bf53af468c9a422c88,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02
...,...,...,...,...,...,...
98172,f7780ea2807db31691e83f0013294035,delivered,2018-07-30 15:22:15,2018-07-30 15:35:16,2018-07-30 15:00:00,2018-08-02 18:32:30
98430,d7646ffe8fdd9e7d9557f9f7cbf04530,delivered,2018-05-04 14:50:37,2018-05-04 15:10:22,2018-05-04 14:48:00,2018-05-08 19:06:42
98672,5ded8a3706eabd813685534724f066de,delivered,2018-07-18 08:46:52,2018-07-18 09:01:48,2018-07-18 08:44:00,2018-07-25 13:53:17
98780,d10046876c7d9f01613da59ffa6cb07f,delivered,2018-07-18 16:14:16,2018-07-18 16:25:17,2018-07-18 15:34:00,2018-07-23 20:46:44


In [86]:
orders.loc[
    customer_before_carrier,
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ],
]

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-02 20:58:43,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-16 12:35:34,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-18 11:48:20,2017-07-20 23:03:42,2017-07-20 18:52:41
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-26 11:41:53,2016-10-25 17:51:46
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-14 20:17:04,2017-06-27 14:51:54,2017-06-26 15:45:35
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-07 21:35:22,2017-08-10 18:28:56,2017-08-10 18:05:38
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-07-26 11:46:34,2017-08-01 18:17:47,2017-07-31 17:49:56
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-08-09 18:18:43,2017-08-01 21:13:01


In [87]:
orders.loc[
    delivered_without_customer_date,
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ],
]

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-30 18:12:23,NaT
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-25 08:05:00,NaT
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-03 13:57:00,NaT
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-03 13:57:00,NaT
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-03 09:28:00,NaT
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaT,NaT
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-12 14:10:00,NaT
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-07-03 19:26:00,NaT


In [88]:
orders.loc[
    not_delivered_with_customer_date,
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ],
]

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-02-20 19:57:13,2018-03-21 22:03:51
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-13 13:36:59,2016-10-16 14:36:59
58266,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-07 14:52:30,2016-10-11 15:07:11,2016-10-14 15:07:11
59332,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-08 20:17:50,2016-10-14 22:45:26,2016-10-19 18:47:43
92636,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-10-03 21:01:41,2016-10-25 12:14:28,2016-11-08 10:58:34
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,canceled,2016-10-09 15:39:56,2016-10-14 10:40:50,2016-11-09 14:53:50


### Investigation Findings

The integrity checks identified several timestamp anomalies.

- 166 orders have a carrier delivery timestamp earlier than the purchase timestamp. These cases are retained as data anomalies because the available data does not provide enough evidence to determine whether the timestamps are incorrect.
- 23 orders have a customer delivery timestamp earlier than the carrier delivery timestamp. These cases are retained and flagged as timestamp inconsistencies.
- 8 orders are marked as delivered but have no customer delivery timestamp.
- 6 canceled orders have a customer delivery timestamp.

No negative values were found for item prices, freight values, or payment values.

## Duplicate & Uniqueness Checks

This section checks for duplicate records and verifies uniqueness assumptions for tables without a single primary key.

We've already looked at `duplicated()` in profiling, but here we want to take the look a little more in-depth.

In [89]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_summary.append(
        {
            "table": name,
            "row_count": len(df),
            "duplicate_rows": df.duplicated().sum(),
        }
    )

duplicate_summary = pd.DataFrame(duplicate_summary)

duplicate_summary

,table,row_count,duplicate_rows
0,olist_customers_dataset,99441,0
1,olist_geolocation_dataset,1000163,261831
2,olist_order_items_dataset,112650,0
3,olist_order_payments_dataset,103886,0
4,olist_order_reviews_dataset,99224,0
5,olist_orders_dataset,99441,0
6,olist_products_dataset,32951,0
7,olist_sellers_dataset,3095,0
8,product_category_name_translation,71,0


### Geolocation

Let's examine this table separately, because we know from profiling that it has a complete duplicate.

In [90]:
geolocation[geolocation.duplicated(keep=False)].sort_values(
    by=geolocation.columns.tolist()
).head(20)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
519,1001,-23.551337,-46.634027,sao paulo,SP
583,1001,-23.551337,-46.634027,sao paulo,SP
818,1001,-23.551337,-46.634027,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
596,1001,-23.550498,-46.634338,sao paulo,SP
639,1001,-23.550498,-46.634338,sao paulo,SP
771,1001,-23.550498,-46.634338,sao paulo,SP
912,1001,-23.550498,-46.634338,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP


But more important than the complete duplicate is to see:
> Can a `geolocation_zip_code_prefix` have multiple coordinates?

In [91]:
geolocation_per_zip = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .size()
    .rename("location_count")
)

geolocation_per_zip.describe()

count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
Name: location_count, dtype: float64

In [92]:
(geolocation_per_zip > 1).sum()

np.int64(17972)

### Reviews

In profiling we saw that `review_id` is not unique in this table. Now we need to understand more precisely how the duplicates are.

In [93]:
review_id_counts = reviews["review_id"].value_counts()

review_id_counts.value_counts().sort_index()

count
1    97621
2      764
3       25
Name: count, dtype: int64

In [94]:
(review_id_counts > 1).sum()

np.int64(789)

Some examples of duplicate review IDs:

In [95]:
duplicate_review_ids = review_id_counts[
    review_id_counts > 1
].index

reviews[
    reviews["review_id"].isin(duplicate_review_ids)
].sort_values("review_id").head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


The goal here is not to remove duplicates; we want to understand whether `review_id` can actually be used as a unique identifier.

### Product Categories

Let's see how many different products belong to the same category:

In [96]:
products_per_category = (
    products.groupby("product_category_name")
    .size()
    .sort_values(ascending=False)
)

products_per_category.head(20)

product_category_name
cama_mesa_banho                      3029
esporte_lazer                        2867
moveis_decoracao                     2657
beleza_saude                         2444
utilidades_domesticas                2335
automotivo                           1900
informatica_acessorios               1639
brinquedos                           1411
relogios_presentes                   1329
telefonia                            1134
bebes                                 919
perfumaria                            868
fashion_bolsas_e_acessorios           849
papelaria                             849
cool_stuff                            789
ferramentas_jardim                    753
pet_shop                              719
eletronicos                           517
construcao_ferramentas_construcao     400
eletrodomesticos                      370
dtype: int64

And the number of products that do not have a category:

In [97]:
products["product_category_name"].isna().sum()

np.int64(610)

This section is more for understanding the structure, not to consider nulls as a problem for now.

### An important point about Translation

Examining the relationship between category and the translation table:

In [98]:
category_translation[
    "product_category_name"
].nunique()

71

In [99]:
products[
    "product_category_name"
].nunique()

73

Next, checking which Product categories are not present in translation:

In [100]:
product_categories = set(
    products["product_category_name"].dropna().unique()
)

translated_categories = set(
    category_translation["product_category_name"].dropna().unique()
)

untranslated_categories = product_categories - translated_categories

untranslated_categories

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}

### Findings

- No complete duplicate records were found except in the geolocation table.
- `geolocation_zip_code_prefix` is not unique and can map to many geographic records. This should be considered when joining geolocation data to other tables.
- `review_id` is not unique in the reviews table. Repeated review IDs are retained for further investigation.
- 610 products have no category assigned.
- Two product categories do not have a corresponding entry in the category translation table.

## Geolocation Structure

The geolocation table contains multiple records for the same ZIP code prefix. This check determines whether these records represent duplicate observations or distinct geographic coordinates.

First, we don't remove complete duplicates; we just create a temporary DataFrame for checking:

In [101]:
geolocation_unique = geolocation.drop_duplicates()

Now the number of records remaining:

In [102]:
len(geolocation), len(geolocation_unique)

(1000163, 738332)

To see how many different coordinates we have for each ZIP prefix:

In [103]:
coordinates_per_zip = (
    geolocation_unique
    .groupby("geolocation_zip_code_prefix")
    .size()
    .rename("coordinate_count")
)

coordinates_per_zip.describe()

count    19015.000000
mean        38.828925
std         50.875735
min          1.000000
25%          8.000000
50%         23.000000
75%         49.000000
max        779.000000
Name: coordinate_count, dtype: float64

In [104]:
(coordinates_per_zip > 1).sum()

np.int64(17823)

Finally, some examples of ZIPs that have multiple coordinates:

In [105]:
coordinates_per_zip[
    coordinates_per_zip > 1
].sort_values(ascending=False).head(10)

geolocation_zip_code_prefix
38400    779
35500    751
11680    727
11740    678
36400    627
38408    621
39400    620
35162    611
37200    596
35900    589
Name: coordinate_count, dtype: int64

### Geolocation Findings

After removing complete duplicate records, most ZIP code prefixes still map to multiple geographic coordinates.

Therefore, `geolocation_zip_code_prefix` should not be treated as a unique key when joining geolocation data to customers or sellers.